<a href="https://colab.research.google.com/github/lcipolina/art/blob/main/test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# THIS IS A PROOF OF CONCEPT NOTEBOOK, IN WHICH OUR CLASSIFICATION MODEL IS EXPLAINED

In [1]:
# Download model from Repo
!git clone https://github.com/lcipolina/art --quiet

In [2]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import copy
import torch
from torch.nn import Linear
import torch.nn.functional as F
from torch import nn
import torch.optim as optim
from torch.optim import lr_scheduler
from tqdm import tqdm
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
%matplotlib inline
from tqdm import tqdm

In [3]:
# Some dependencies
os.environ['TORCH'] = torch.__version__
print(torch.__version__)

!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q git+https://github.com/pyg-team/pytorch_geometric.git

1.11.0+cu113
     |████████████████████████████████| 7.9 MB 8.1 MB/s 
     |████████████████████████████████| 3.5 MB 8.3 MB/s 


In [4]:
from torch_geometric.nn import HeteroConv, GATConv
from torch_geometric.seed import seed_everything
seed_everything(1)

In [5]:
%cd art
from load_dataset.artgraph import ArtGraph
from model import ModelClassification

/content/art


In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## UTILS

In [7]:
class ClassificationDataSet(torch.utils.data.Dataset):
    def __init__(self, graph, sub = 'artwork', obj = 'style'):
        self.graph = graph
        self.data = graph[sub, obj].edge_label_index.T[graph[sub, obj].edge_label==1]
        self.sub = sub
        self.obj = obj
    
    def __len__(self):
        return self.data.shape[0]
    
    def __getitem__(self, idx):
        return self.data[idx,0], self.graph[self.sub].x[self.data[idx,0]], self.data[idx, 1]
    
    def get_links(self):
        return pd.DataFrame(self.data.cpu().numpy(), columns = [self.sub, self.obj])

Let's see how the model works

In [8]:
#%cd art
model = torch.load('play_model.pt')
model.to(device)

ModelClassification(
  (encoder): MultiGNNEncoder(
    (activation): Tanh()
    (encoders): ModuleDict(
      (0): HeteroConv(num_relations=1)
    )
  )
  (decoder): Head(
    (head): Sequential(
      (0): Linear(in_features=2432, out_features=1216, bias=True)
      (1): LeakyReLU(negative_slope=0.1, inplace=True)
      (2): Dropout(p=0.25, inplace=False)
      (3): Linear(in_features=1216, out_features=608, bias=True)
      (4): LeakyReLU(negative_slope=0.1, inplace=True)
      (5): Dropout(p=0.25, inplace=False)
      (6): Linear(in_features=608, out_features=304, bias=True)
      (7): LeakyReLU(negative_slope=0.1, inplace=True)
      (8): Dropout(p=0.25, inplace=False)
      (9): Linear(in_features=304, out_features=152, bias=True)
      (10): LeakyReLU(negative_slope=0.1, inplace=True)
      (11): Dropout(p=0.25, inplace=False)
      (12): Linear(in_features=152, out_features=18, bias=True)
    )
  )
)

The model consists of a single encoder layer, in which a style is encoded as an aggregation of multiple artworks

In [9]:
model.encoder.encoders['0']._modules

OrderedDict([('convs', ModuleDict(
                (artwork__hasgenre__genre): GATConv((-1, -1), 128, heads=1)
              ))])

Load test graph

Note that the attribute `edge label` and `edge label index` refer to test pairs. 

In [12]:
#FIRST: upload the file manually (too big for GitHub)
test_data = torch.load(r'dataset_full_conf/test_data_genre_vit_fine-tuning.pt')
test_data.to(device)

HeteroData(
  artwork={ x=[116475, 128] },
  artist={ x=[2501, 1] },
  gallery={ x=[1099, 1] },
  city={ x=[596, 1] },
  country={ x=[58, 1] },
  style={ x=[32, 1] },
  period={ x=[186, 1] },
  genre={ x=[18, 1] },
  serie={ x=[823, 1] },
  tag={ x=[5424, 1] },
  media={ x=[167, 1] },
  subject={ x=[6985, 1] },
  training_node={ x=[268, 1] },
  field={ x=[54, 1] },
  movement={ x=[243, 1] },
  people={ x=[109, 1] },
  emotion={ x=[9, 1] },
  (artist, belongstofield, field)={ edge_index=[2, 987] },
  (artist, belongstomovement, movement)={ edge_index=[2, 1056] },
  (artist, haspatron, people)={ edge_index=[2, 124] },
  (artist, hassubject, subject)={ edge_index=[2, 21054] },
  (artist, relatedtoschool, training_node)={ edge_index=[2, 498] },
  (artist, trainedby, artist)={ edge_index=[2, 94] },
  (artwork, about, tag)={ edge_index=[2, 241203] },
  (artwork, createdby, artist)={ edge_index=[2, 81532] },
  (artwork, elicit, emotion)={ edge_index=[2, 45649] },
  (artwork, hasgenre, genre)=

In [13]:
batch_size = 128
test_dataset = ClassificationDataSet(test_data, obj = 'genre')
test_loader = DataLoader(test_dataset, batch_size = batch_size, shuffle = True, drop_last = False)

In [14]:
with torch.no_grad():
    for idxs, imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(test_loader.dataset.graph.x_dict, test_loader.dataset.graph.edge_index_dict, imgs)#having predictions
        outputs = nn.Softmax(dim = 1)(outputs)#applying softmax activation function
        _, preds = torch.max(nn.Softmax(dim = 1)(outputs), 1)#having predicted style for each artwork
        break

TypeError: ignored

Let's see how the model classifies

In [15]:
#building results table
res_df = pd.DataFrame(columns = ['artwork', 'genre', 'predicted_genre'])
res_df['artwork'] = idxs.cpu().numpy()
res_df['genre'] = labels.cpu().numpy()
res_df['predicted_genre'] = preds.cpu().numpy()
res_df # the artwork `x` is of genre `y` and the predicted one is `z`

NameError: ignored

In [16]:
#getting misclassified artwork
mis_entry = res_df[res_df['genre'] != res_df['predicted_genre']].sample(n = 1, random_state = 42)
mis_entry

,artwork,genre,predicted_genre
55,85205,16,NaN


In [17]:
#getting a random artwork correctly classified
right_entry = res_df[res_df['genre'] == res_df['predicted_genre']].sample(n = 1, random_state = 42)
right_entry

ValueError: ignored

In [ ]:
artwork2name = pd.read_csv(r'artgraph2bestemotions/mapping/artwork_entidx2name.csv', header = None, names = ['idx', 'name'])
artwork2name

In [ ]:
genre2name = pd.read_csv(r'artgraph2bestemotions/mapping/genre_entidx2name.csv', header = None, names = ['idx', 'name'])
genre2name

In [ ]:
#show misclassified artwork
from PIL import Image
img_name = artwork2name.loc[mis_entry.iloc[0].artwork]["name"]
img = Image.open(fr'images-resized/{img_name}')
plt.imshow(img)
plt.title(f'Right classification of image: {img_name}')
plt.xlabel(f'''Classified genre: {genre2name.loc[mis_entry.iloc[0].predicted_genre]['name']}\n
           Correct genre: {genre2name.loc[mis_entry.iloc[0]['genre']]['name']}''')
plt.show()

In [ ]:
#correctly classified artwork
img_name = artwork2name.loc[right_entry.iloc[0].artwork]["name"]
img = Image.open(fr'images-resized/{img_name}')
plt.imshow(img)
plt.title(f'Misclassification of image: {img_name}')
plt.xlabel(f'''Classified genre: {genre2name.loc[right_entry.iloc[0].predicted_genre]['name']}\n
           Correct genre: {genre2name.loc[right_entry.iloc[0]['genre']]['name']}''')
plt.show()

As we can see, this domain is very tricky because the distance between genres (in this case) is not equal between each possible pair of genres (that is to say `figurative` **is really similar to** `abstract`)